In [1]:
import pandas as pd
import numpy as np
import os
import sys
notebook_dir = os.path.abspath(os.path.dirname(''))
project_root = os.path.dirname(notebook_dir)
sys.path.append(project_root)
import pandas as pd
from src.utils.db_utils import get_connection, execute_query
import xgboost as xgb

from src.features.team_level_features import team_level_features_class
from src.models import xgboost_randomforrest_model
from src.utils.y_variable import build_spread_variable

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from src.utils import config

In [2]:
nfl_model=xgboost_randomforrest_model.NFLPredictor()
nfl_model.build_x_y_variable()

Successfully connected to the database!
Successfully connected to the database!
creating efficiency stats


100%|██████████| 5/5 [00:00<00:00, 320.06it/s]


building team stats


100%|██████████| 25/25 [00:14<00:00,  1.77it/s]


building X Y Dataframes


100%|██████████| 100/100 [00:48<00:00,  2.08it/s]


In [3]:
x=nfl_model.df_x.iloc[:,:-1].dropna()
y=nfl_model.df_y.loc[x.index, "binary_spread_label"]

In [4]:
nfl_model.df_y

,spread,total points,binary_spread_label,binary_ou_label,gamesummaryid
2023_1_0,-2,70,1,1,gs-202301MIALAC
2023_1_1,-1,33,1,0,gs-202301LVRDEN
2023_1_2,-17,43,0,0,gs-202301LARSEA
2023_1_3,14,34,1,0,gs-202301CARATL
2023_1_4,16,34,1,0,gs-202301HOUBAL
...,...,...,...,...,...
2024_20_2,2,52,0,1,gs-202420BALBUF
2024_20_3,6,50,1,1,gs-202420LARPHI
2024_21_0,3,61,1,1,gs-202421BUFKAN
2024_21_1,32,78,1,1,gs-202421WASPHI


In [5]:
for i_model in range(len(config.model_config.classification_models)):
    label=config.model_config.classification_models[i_model]
    model_name=config.model_config.classification_models_names[i_model]
    x=nfl_model.df_x.iloc[:,:-1].dropna()
    y=nfl_model.df_y.loc[x.index, label]
    nfl_model.build_xg_boost_model(x,y,config.model_config.param_dist,model_name, xgb.XGBClassifier)

c:\Users\sam sheeran\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:952: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(
c:\Users\sam sheeran\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:952: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


In [6]:
for i_model in range(len(config.model_config.regression_models)):
    label=config.model_config.regression_models[i_model]
    model_name=config.model_config.regression_models_names[i_model]
    x=nfl_model.df_x.iloc[:,:-1].dropna()
    y=nfl_model.df_y.loc[x.index, label]
    nfl_model.build_xg_boost_model(x,y,config.model_config.param_dist,model_name, xgb.XGBRegressor)

In [7]:
nfl_model.model_prediction(nfl_model.xg_model, x=nfl_model.x_test.to_numpy(),plot_feature_importance=True, plot_scatter=True)

AttributeError: 'NFLPredictor' object has no attribute 'xg_model'

In [ ]:
current_game_stats=nfl_model.build_single_game_features("PHI", "KAN")

model=nfl_model.xg_model.fit(nfl_model.x_y.iloc[:,:-1].to_numpy(), nfl_model.x_y.iloc[:,-1])
model.predict(current_game_stats.iloc[:,:-1].to_numpy())

array([3.5566726], dtype=float32)

In [ ]:
model.predict(nfl_model.x_y.iloc[-2:,:-1].to_numpy())

array([1.0102892, 0.7356641], dtype=float32)